# Getting a Sample of 3000 using those with NER recognised Entities

In [1]:
import random
import pandas as pd

def generate_random_list(max_num, list_size=3000, seed=1):
    """
    Generate a list of unique random integers from 0 to max_num - 1 (no duplicates).

    Parameters:
        max_num (int): The exclusive upper limit for random numbers.
        list_size (int): Number of random integers to generate.
        seed (int): Random seed for reproducibility.

    Returns:
        List[int]: A list of unique random integers.
    """
    if list_size > max_num:
        raise ValueError("list_size cannot be greater than max_num when sampling without replacement.")

    random.seed(seed)
    return random.sample(range(0, max_num), list_size)


In [2]:
#Load dataset
entity_df = pd.read_parquet('named_entities.parquet', engine='pyarrow')
df = pd.read_parquet('cleaned_news_data.parquet', engine = 'pyarrow')
df = df[df['clean_full_text'].apply(len) > 0]
df = df.reset_index(drop = True)

ner_index = entity_df[(entity_df['Persons'].apply(len) != 0) | (entity_df['Organizations'].apply(len) != 0)].index

# Final news data
final_news_df = df.loc[ner_index]
final_news_df.to_parquet('final_cleaned_news_data.parquet', engine = 'pyarrow', index = False)

# We do a random sample for this analysis as it takes too long
sampled_index = generate_random_list(max_num = len(final_news_df), list_size = 3000)
final_news_df.iloc[sampled_index].to_parquet('final_cleaned_news_data_sampled.parquet', engine = 'pyarrow', index = False)

# The above 3000 samples will be labelled